In [25]:
import sympy as sp
import sympy.diffgeom as dg

from sympy.printing.str import StrPrinter

In [41]:
# these expressions are error prone if entered by hand so we just generate it
# to be used directly in the code (with some necessary modifications due to the
# latex naming of the symbols for presentation in this notebook)


class DPrinter(StrPrinter):
    def _print_Function(self, expr):
        # remove (t)
        return expr.func.__name__

    def _print_Derivative(self, expr):
        func = expr.expr
        if func.is_Function:
            name = func.func.__name__
            order = sum(count for _, count in expr.variable_count)
            return "d_" * order + name
        return super()._print_Derivative(expr)


printer = DPrinter()

In [27]:
state_mfld = dg.Manifold("M", 5)
state_mfld_patch = dg.Patch("P", state_mfld)

x, y, theta, v, omega = sp.symbols(r"x,y,\theta,v,\omega", real=True)

state_mfld_coords = dg.CoordSystem(
    "StateSpace", state_mfld_patch, (x, y, theta, v, omega)
)

(x_sc, y_sc, theta_sc, v_sc, omega_sc) = state_mfld_coords.base_scalars()
(x_vec, y_vec, theta_vec, v_vec, omega_vec) = state_mfld_coords.base_vectors()

In [28]:
# define the vector fields of the system

f = (
    v_sc * sp.cos(theta_sc) * x_vec
    + v_sc * sp.sin(theta_sc) * y_vec
    + omega_sc * theta_vec
)

f1 = v_vec
f2 = omega_vec

u_f, u_t = sp.symbols("u_f,u_t", real=True)

display(f, f1, f2)
display(u_f, u_t)

sin(\theta)*v*e_y + cos(\theta)*v*e_x + \omega*e_\theta

e_v

e_\omega

u_f

u_t

In [29]:
# define the variables associated with the keep out region
t = sp.symbols("t", real=True)

x_ko = sp.Function("x_ko")(t)
y_ko = sp.Function("y_ko")(t)
r_ko = sp.Function("r_ko")(t)

ko_cbf = sp.sqrt((x_sc - x_ko) ** 2 + (y_sc - y_ko) ** 2) - r_ko
display(ko_cbf)

sqrt((-x_ko(t) + x)**2 + (-y_ko(t) + y)**2) - r_ko(t)

In [30]:
# generates the condition for the HOCBF


def _repeated_lie_deriv(f, h, n):
    if n == 0:
        return h
    return _repeated_lie_deriv(f, dg.LieDerivative(f, h), n - 1)


def gen_hocbf_condition(dyn_f, dyn_gs, us, m, h, t, alpha_fns):
    lie_m_fh = _repeated_lie_deriv(dyn_f, h, m)
    lie_g_fh = sum(
        dg.LieDerivative(g, _repeated_lie_deriv(f, h, m - 1)) * u
        for (g, u) in zip(dyn_gs, us)
    )
    dmh_dtm = sp.diff(h, t, m)

    alpha_fns.insert(0, lambda phi: 1.0 * phi)  # dummy to ensure correct indexing

    phi_fns = [h]
    for i in range(1, m):
        phi_fns.append(sp.diff(phi_fns[i - 1], t) + alpha_fns[i](phi_fns[i - 1]))

    o_h = 0
    for i in range(1, m - 1):
        # same thing here with the alpha indexing
        alpha_phi_fn = alpha_fns[m - i](phi_fns[m - i - 1])
        o_h += _repeated_lie_deriv(f, alpha_phi_fn, i) + sp.diff(alpha_phi_fn, t, i)

    final_alpha_phi = alpha_fns[m](phi_fns[m - 1])

    return lie_m_fh + lie_g_fh + dmh_dtm + o_h + final_alpha_phi


# assuming the use of linear extended K_infty functions
k1, k2 = sp.symbols("k1,k2", nonneg=True, real=True)

ko_hocbf = (
    gen_hocbf_condition(
        f, [f1, f2], [u_f, u_t], 2, ko_cbf, t, [lambda x: k1 * x, lambda x: k2 * x]
    )
    .nsimplify()
    .simplify()
)

display(ko_hocbf)  # ouch

(-((((x_ko(t) - x)*cos(\theta) + (y_ko(t) - y)*sin(\theta))*(x_ko(t) - x) - ((x_ko(t) - x)**2 + (y_ko(t) - y)**2)*cos(\theta))*cos(\theta) + (((x_ko(t) - x)*cos(\theta) + (y_ko(t) - y)*sin(\theta))*(y_ko(t) - y) - ((x_ko(t) - x)**2 + (y_ko(t) - y)**2)*sin(\theta))*sin(\theta))*((x_ko(t) - x)**2 + (y_ko(t) - y)**2)**(5/2)*v**2 - ((x_ko(t) - x)*Derivative(x_ko(t), t) + (y_ko(t) - y)*Derivative(y_ko(t), t))**2*((x_ko(t) - x)**2 + (y_ko(t) - y)**2)**(5/2) + ((x_ko(t) - x)**2 + (y_ko(t) - y)**2)**(7/2)*(-k2*((-k1*(sqrt((x_ko(t) - x)**2 + (y_ko(t) - y)**2) - r_ko(t)) + Derivative(r_ko(t), t))*sqrt((x_ko(t) - x)**2 + (y_ko(t) - y)**2) - (x_ko(t) - x)*Derivative(x_ko(t), t) - (y_ko(t) - y)*Derivative(y_ko(t), t)) - u_f*((x_ko(t) - x)*cos(\theta) + (y_ko(t) - y)*sin(\theta)) + ((x_ko(t) - x)*sin(\theta) + (-y_ko(t) + y)*cos(\theta))*v*\omega + (x_ko(t) - x)*Derivative(x_ko(t), (t, 2)) + (y_ko(t) - y)*Derivative(y_ko(t), (t, 2)) + Derivative(x_ko(t), t)**2 + Derivative(y_ko(t), t)**2) - ((x_ko(t

In [31]:
# devise some simplifications that might help

x_err_sym, y_err_sym, r_sqr_err_sym, r_1_sym, r_2_sym, r_3_sym, r_4_sym, r_5_sym = (
    sp.symbols("x_err,y_err,r^2_err,r_1,r_2,r_3,r_4,r_5", real=True)
)

x_err = x_ko - x_sc
y_err = y_ko - y_sc
r_sqr_err = x_err_sym**2 + y_err_sym**2

r_1 = r_sqr_err_sym ** (sp.Rational(7 / 2)) * x_err_sym
r_2 = r_sqr_err_sym ** (sp.Rational(7 / 2)) * y_err_sym
r_3 = r_sqr_err_sym ** (sp.Rational(5 / 2)) * x_err_sym**2
r_4 = r_sqr_err_sym ** (sp.Rational(5 / 2)) * y_err_sym**2
r_5 = r_sqr_err_sym ** (sp.Rational(5 / 2)) * x_err_sym * y_err_sym

ko_hocbf_cleaned = (
    (
        ko_hocbf.subs(x_err, x_err_sym)
        .subs(y_err, y_err_sym)
        .subs(r_sqr_err, r_sqr_err_sym)
    )
    .simplify()
    .factor()
    .subs(r_1, r_1_sym)
    .subs(r_2, r_2_sym)
    .subs(r_3, r_3_sym)
    .subs(r_4, r_4_sym)
    .subs(r_5, r_5_sym)
).factor()
display(ko_hocbf_cleaned)

-(-2*k1*k2*r^2_err**(9/2) + 2*k1*k2*r^2_err**4*r_ko(t) + 2*k2*r^2_err**4*Derivative(r_ko(t), t) - 2*k2*r_1*Derivative(x_ko(t), t) - 2*k2*r_2*Derivative(y_ko(t), t) - 2*r^2_err**(7/2)*v**2 - 2*r^2_err**(7/2)*Derivative(x_ko(t), t)**2 - 2*r^2_err**(7/2)*Derivative(y_ko(t), t)**2 + 2*r^2_err**4*Derivative(r_ko(t), (t, 2)) + 2*r_1*u_f*cos(\theta) - 2*r_1*sin(\theta)*v*\omega - 2*r_1*Derivative(x_ko(t), (t, 2)) + 2*r_2*u_f*sin(\theta) + 2*r_2*cos(\theta)*v*\omega - 2*r_2*Derivative(y_ko(t), (t, 2)) + r_3*cos(2*\theta)*v**2 + r_3*v**2 + 2*r_3*Derivative(x_ko(t), t)**2 - r_4*cos(2*\theta)*v**2 + r_4*v**2 + 2*r_4*Derivative(y_ko(t), t)**2 + 2*r_5*sin(2*\theta)*v**2 + 4*r_5*Derivative(x_ko(t), t)*Derivative(y_ko(t), t))/(2*r^2_err**4)

In [32]:
# final HOCBF (to use as condition in optimization problem)
ko_hocbf_condition = ko_hocbf_cleaned >= 0
display(ko_hocbf_condition)

display(sp.Eq(x_err_sym, x_err))
display(sp.Eq(y_err_sym, y_err))
display(sp.Eq(r_sqr_err_sym, r_sqr_err))
display(sp.Eq(r_1_sym, r_1))
display(sp.Eq(r_2_sym, r_2))
display(sp.Eq(r_3_sym, r_3))
display(sp.Eq(r_4_sym, r_4))
display(sp.Eq(r_5_sym, r_5))

sp.print_latex(ko_hocbf_condition)

-(-2*k1*k2*r^2_err**(9/2) + 2*k1*k2*r^2_err**4*r_ko(t) + 2*k2*r^2_err**4*Derivative(r_ko(t), t) - 2*k2*r_1*Derivative(x_ko(t), t) - 2*k2*r_2*Derivative(y_ko(t), t) - 2*r^2_err**(7/2)*v**2 - 2*r^2_err**(7/2)*Derivative(x_ko(t), t)**2 - 2*r^2_err**(7/2)*Derivative(y_ko(t), t)**2 + 2*r^2_err**4*Derivative(r_ko(t), (t, 2)) + 2*r_1*u_f*cos(\theta) - 2*r_1*sin(\theta)*v*\omega - 2*r_1*Derivative(x_ko(t), (t, 2)) + 2*r_2*u_f*sin(\theta) + 2*r_2*cos(\theta)*v*\omega - 2*r_2*Derivative(y_ko(t), (t, 2)) + r_3*cos(2*\theta)*v**2 + r_3*v**2 + 2*r_3*Derivative(x_ko(t), t)**2 - r_4*cos(2*\theta)*v**2 + r_4*v**2 + 2*r_4*Derivative(y_ko(t), t)**2 + 2*r_5*sin(2*\theta)*v**2 + 4*r_5*Derivative(x_ko(t), t)*Derivative(y_ko(t), t))/(2*r^2_err**4) >= 0

Eq(x_err, x_ko(t) - x)

Eq(y_err, y_ko(t) - y)

Eq(r^2_err, x_err**2 + y_err**2)

Eq(r_1, r^2_err**(7/2)*x_err)

Eq(r_2, r^2_err**(7/2)*y_err)

Eq(r_3, r^2_err**(5/2)*x_err**2)

Eq(r_4, r^2_err**(5/2)*y_err**2)

Eq(r_5, r^2_err**(5/2)*x_err*y_err)

- \frac{- 2 k_{1} k_{2} \left(r^{2}_{err}\right)^{\frac{9}{2}} + 2 k_{1} k_{2} \left(r^{2}_{err}\right)^{4} r_{ko}{\left(t \right)} + 2 k_{2} \left(r^{2}_{err}\right)^{4} \frac{d}{d t} r_{ko}{\left(t \right)} - 2 k_{2} r_{1} \frac{d}{d t} x_{ko}{\left(t \right)} - 2 k_{2} r_{2} \frac{d}{d t} y_{ko}{\left(t \right)} - 2 \left(r^{2}_{err}\right)^{\frac{7}{2}} \mathbf{v}^{2} - 2 \left(r^{2}_{err}\right)^{\frac{7}{2}} \left(\frac{d}{d t} x_{ko}{\left(t \right)}\right)^{2} - 2 \left(r^{2}_{err}\right)^{\frac{7}{2}} \left(\frac{d}{d t} y_{ko}{\left(t \right)}\right)^{2} + 2 \left(r^{2}_{err}\right)^{4} \frac{d^{2}}{d t^{2}} r_{ko}{\left(t \right)} + 2 r_{1} u_{f} \cos{\left(\mathbf{\theta} \right)} - 2 r_{1} \sin{\left(\mathbf{\theta} \right)} \mathbf{v} \mathbf{\omega} - 2 r_{1} \frac{d^{2}}{d t^{2}} x_{ko}{\left(t \right)} + 2 r_{2} u_{f} \sin{\left(\mathbf{\theta} \right)} + 2 r_{2} \cos{\left(\mathbf{\theta} \right)} \mathbf{v} \mathbf{\omega} - 2 r_{2} \frac{d^{2}}{d t^{2}} y_{ko}{\left

In [42]:
print(printer.doprint(ko_hocbf_cleaned))

-(-2*k1*k2*r^2_err**(9/2) + 2*k1*k2*r^2_err**4*r_ko + 2*k2*r^2_err**4*d_r_ko - 2*k2*r_1*d_x_ko - 2*k2*r_2*d_y_ko - 2*r^2_err**(7/2)*v**2 - 2*r^2_err**(7/2)*d_x_ko**2 - 2*r^2_err**(7/2)*d_y_ko**2 + 2*r^2_err**4*d_d_r_ko + 2*r_1*u_f*cos - 2*r_1*sin*v*\omega - 2*r_1*d_d_x_ko + 2*r_2*u_f*sin + 2*r_2*cos*v*\omega - 2*r_2*d_d_y_ko + r_3*cos*v**2 + r_3*v**2 + 2*r_3*d_x_ko**2 - r_4*cos*v**2 + r_4*v**2 + 2*r_4*d_y_ko**2 + 2*r_5*sin*v**2 + 4*r_5*d_x_ko*d_y_ko)/(2*r^2_err**4)


In [14]:
ko_hocbf_cleaned.expand().coeff(u_f)

-r_1*cos(\theta)/r^2_err**4 - r_2*sin(\theta)/r^2_err**4

The following computes the gradient and part of the covariant hessian necessary for implementation of the GHOCBF in the product manifold formulation.

In [15]:
ko_hocbf

(-((((x_ko(t) - x)*cos(\theta) + (y_ko(t) - y)*sin(\theta))*(x_ko(t) - x) - ((x_ko(t) - x)**2 + (y_ko(t) - y)**2)*cos(\theta))*cos(\theta) + (((x_ko(t) - x)*cos(\theta) + (y_ko(t) - y)*sin(\theta))*(y_ko(t) - y) - ((x_ko(t) - x)**2 + (y_ko(t) - y)**2)*sin(\theta))*sin(\theta))*((x_ko(t) - x)**2 + (y_ko(t) - y)**2)**(5/2)*v**2 - ((x_ko(t) - x)*Derivative(x_ko(t), t) + (y_ko(t) - y)*Derivative(y_ko(t), t))**2*((x_ko(t) - x)**2 + (y_ko(t) - y)**2)**(5/2) + ((x_ko(t) - x)**2 + (y_ko(t) - y)**2)**(7/2)*(-k2*((-k1*(sqrt((x_ko(t) - x)**2 + (y_ko(t) - y)**2) - r_ko(t)) + Derivative(r_ko(t), t))*sqrt((x_ko(t) - x)**2 + (y_ko(t) - y)**2) - (x_ko(t) - x)*Derivative(x_ko(t), t) - (y_ko(t) - y)*Derivative(y_ko(t), t)) - u_f*((x_ko(t) - x)*cos(\theta) + (y_ko(t) - y)*sin(\theta)) + ((x_ko(t) - x)*sin(\theta) + (-y_ko(t) + y)*cos(\theta))*v*\omega + (x_ko(t) - x)*Derivative(x_ko(t), (t, 2)) + (y_ko(t) - y)*Derivative(y_ko(t), (t, 2)) + Derivative(x_ko(t), t)**2 + Derivative(y_ko(t), t)**2) - ((x_ko(t

In [16]:
# replace the scalar functions in the expression with the symbols so we can evaluate regular derivatives for
# the gradient and the necessary part of the covariant hessian
ko_hocbf_symbs = (
    ko_hocbf.subs(x_sc, x)
    .subs(y_sc, y)
    .subs(theta_sc, theta)
    .subs(v_sc, v)
    .subs(omega_sc, omega)
)
ko_hocbf_symbs

(-v**2*(((-x + x_ko(t))*((-x + x_ko(t))*cos(\theta) + (-y + y_ko(t))*sin(\theta)) - ((-x + x_ko(t))**2 + (-y + y_ko(t))**2)*cos(\theta))*cos(\theta) + ((-y + y_ko(t))*((-x + x_ko(t))*cos(\theta) + (-y + y_ko(t))*sin(\theta)) - ((-x + x_ko(t))**2 + (-y + y_ko(t))**2)*sin(\theta))*sin(\theta))*((-x + x_ko(t))**2 + (-y + y_ko(t))**2)**(5/2) - ((-x + x_ko(t))*Derivative(x_ko(t), t) + (-y + y_ko(t))*Derivative(y_ko(t), t))**2*((-x + x_ko(t))**2 + (-y + y_ko(t))**2)**(5/2) + ((-x + x_ko(t))**2 + (-y + y_ko(t))**2)**(7/2)*(\omega*v*((-x + x_ko(t))*sin(\theta) + (y - y_ko(t))*cos(\theta)) - k2*(-(-x + x_ko(t))*Derivative(x_ko(t), t) - (-y + y_ko(t))*Derivative(y_ko(t), t) + (-k1*(sqrt((-x + x_ko(t))**2 + (-y + y_ko(t))**2) - r_ko(t)) + Derivative(r_ko(t), t))*sqrt((-x + x_ko(t))**2 + (-y + y_ko(t))**2)) - u_f*((-x + x_ko(t))*cos(\theta) + (-y + y_ko(t))*sin(\theta)) + (-x + x_ko(t))*Derivative(x_ko(t), (t, 2)) + (-y + y_ko(t))*Derivative(y_ko(t), (t, 2)) + Derivative(x_ko(t), t)**2 + Derivativ

In [17]:
# note that as the product manifold optimization requires the optimization
# variables to be the first set of coordinates then we construct the product
# coordinate system as the following

prod_mfld_coords = sp.Matrix([u_f, u_t, x, y, theta, v, omega])
prod_mfld_coords

Matrix([
[   u_f],
[   u_t],
[     x],
[     y],
[\theta],
[     v],
[\omega]])

In [18]:
ko_hocbf_symbs_grad = sp.Matrix([ko_hocbf_symbs]).jacobian(prod_mfld_coords)
ko_hocbf_symbs_grad.T

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [19]:
ko_hocbf_symbs_grad[2]
sp.together(ko_hocbf_symbs_grad)

Matrix([[(-(-x + x_ko(t))*cos(\theta) - (-y + y_ko(t))*sin(\theta))/sqrt((-x + x_ko(t))**2 + (-y + y_ko(t))**2), 0, (8*(-x + x_ko(t))*(-v**2*(((-x + x_ko(t))*((-x + x_ko(t))*cos(\theta) + (-y + y_ko(t))*sin(\theta)) - ((-x + x_ko(t))**2 + (-y + y_ko(t))**2)*cos(\theta))*cos(\theta) + ((-y + y_ko(t))*((-x + x_ko(t))*cos(\theta) + (-y + y_ko(t))*sin(\theta)) - ((-x + x_ko(t))**2 + (-y + y_ko(t))**2)*sin(\theta))*sin(\theta))*((-x + x_ko(t))**2 + (-y + y_ko(t))**2)**(5/2) - ((-x + x_ko(t))*Derivative(x_ko(t), t) + (-y + y_ko(t))*Derivative(y_ko(t), t))**2*((-x + x_ko(t))**2 + (-y + y_ko(t))**2)**(5/2) + ((-x + x_ko(t))**2 + (-y + y_ko(t))**2)**(7/2)*(\omega*v*((-x + x_ko(t))*sin(\theta) + (y - y_ko(t))*cos(\theta)) - k2*(-(-x + x_ko(t))*Derivative(x_ko(t), t) - (-y + y_ko(t))*Derivative(y_ko(t), t) + (-k1*(sqrt((-x + x_ko(t))**2 + (-y + y_ko(t))**2) - r_ko(t)) + Derivative(r_ko(t), t))*sqrt((-x + x_ko(t))**2 + (-y + y_ko(t))**2)) - u_f*((-x + x_ko(t))*cos(\theta) + (-y + y_ko(t))*sin(\the

In [20]:
# use the same previous simplifications to clean up the expressions (but we
# have to redefine them here as they originally used the scalar functions)
x_err = x_ko - x
y_err = y_ko - y
r_sqr_err = x_err_sym**2 + y_err_sym**2

(
    r_1_sym,
    r_2_sym,
    r_3_sym,
    r_4_sym,
    r_5_sym,
    r_6_sym,
    r_7_sym,
    r_8_sym,
    r_9_sym,
    r_10_sym,
    r_11_sym,
    r_12_sym,
    r_13_sym,
    r_14_sym,
    r_15_sym,
) = sp.symbols(
    "r_1,r_2,r_3,r_4,r_5,r_6,r_7,r_8,r_9,r_10,r_11,r_12,r_13,r_14,r_15", real=True
)

r_1 = r_sqr_err_sym * x**2
r_2 = r_sqr_err_sym * y**2
r_3 = r_sqr_err_sym * x * y
r_4 = x**2 * y
r_5 = x * y**2

r_6 = x_ko**2
r_7 = y_ko**2
r_8 = x_ko * y_ko
r_9 = x_ko**2 * y_ko
r_10 = x_ko * y_ko**2

r_11 = x * y_ko
r_12 = y * x_ko
r_13 = x * y_ko**2
r_14 = y * x_ko**2
r_15 = x_ko**2 * y_ko**2

# r_4 = x_ko**2
# r_5 = y_ko**2
# r_6 = x_ko * y_ko

simpl_grad = sp.factor_terms(sp.together(sp.expand_mul(ko_hocbf_symbs_grad)))

simpl_grad = sp.factor_terms(
    sp.together(
        simpl_grad.subs(x_err, x_err_sym)
        .subs(y_err, y_err_sym)
        .subs(r_sqr_err, r_sqr_err_sym)
    )
)

simpl_grad = (
    simpl_grad
    # regular
    .subs(r_1, r_1_sym)
    .subs(r_2, r_2_sym)
    .subs(r_3, r_3_sym)
    .subs(r_4, r_4_sym)
    .subs(r_5, r_5_sym)
    # ko
    .subs(r_6, r_6_sym)
    .subs(r_7, r_7_sym)
    .subs(r_8, r_8_sym)
    .subs(r_9, r_9_sym)
    .subs(r_10, r_10_sym)
    # ko and regular
    .subs(r_11, r_11_sym)
    .subs(r_12, r_12_sym)
    .subs(r_13, r_13_sym)
    .subs(r_14, r_14_sym)
    .subs(r_15, r_15_sym)
    # .subs(r_6, r_6_sym)
)


def final_cleanup(expr):
    factored = sp.factor_terms(sp.together(sp.factor_terms(expr)))
    collected = sp.collect(
        factored, [v * sp.sin(theta), v * sp.cos(theta), sp.sin(theta), sp.cos(theta)]
    )

    return collected


simpl_grad = simpl_grad.applyfunc(final_cleanup)

simpl_grad.T

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [21]:
print(printer.doprint(simpl_grad.T))

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [22]:
ko_hocbf_symbs_hess = ko_hocbf_symbs_grad.jacobian(prod_mfld_coords)
ko_hocbf_symbs_hess

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                   0, 0,                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [23]:
simpl_hess = sp.factor_terms(sp.together(sp.expand_mul(ko_hocbf_symbs_hess)))

simpl_hess = sp.factor_terms(
    sp.together(
        simpl_hess.subs(x_err, x_err_sym)
        .subs(y_err, y_err_sym)
        .subs(r_sqr_err, r_sqr_err_sym)
    )
)

simpl_hess = (
    simpl_hess.subs(r_1, r_1_sym)
    .subs(r_2, r_2_sym)
    .subs(r_3, r_3_sym)
    .subs(r_4, r_4_sym)
    .subs(r_5, r_5_sym)
    .subs(r_6, r_6_sym)
)

simpl_hess = simpl_hess.applyfunc(final_cleanup)

# the hessian is symmetric so we'll only generate the upper-right and use the
# transpose to generate the full hessian later on
simpl_hess = sp.MutableDenseMatrix(simpl_hess)
for i in range(simpl_hess.shape[0]):
    for j in range(simpl_hess.shape[1]):
        if j < i:
            simpl_hess[i, j] = 0

simpl_hess

Matrix([
[0, 0,                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [24]:
print(printer.doprint(simpl_hess))

Matrix([
[0, 0,                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         